# IMPORT DATASETS 

In [2]:
!pip install datasets

In [3]:
pip install -U datasets

Note: you may need to restart the kernel to use updated packages.


In [4]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

/Users/rhythemsabharwalgmail.com/Desktop/SLM/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Exploration

In [5]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [6]:
ds["train"].column_names

['text']

In [7]:
ds["train"].select(range(5)).to_pandas()

,text
0,"One day, a little girl named Lily found a need..."
1,"Once upon a time, there was a little car named..."
2,"One day, a little fish named Fin was swimming ..."
3,"Once upon a time, in a land full of trees, the..."
4,"Once upon a time, there was a little girl name..."


In [8]:
len(ds["train"])

2119719

In [9]:
lengths = [len(x["text"]) for x in ds["train"].select(range(1000))]

In [10]:
import numpy as np

print(np.mean(lengths))
print(np.max(lengths))
print(np.min(lengths))

941.64
4123
274


In [11]:
sum(x["text"] is None for x in ds["train"])

0

In [12]:
sum(len(x["text"]) == 0 for x in ds["train"])

230

# Tokenize the Dataset

## In this step, we will:

### (1) Convert the text into token IDs.
Language models cannot process raw text directly, so each story is converted into a sequence of numerical token IDs using a tokenizer.

### (2) Save the token IDs into `train.bin` and `validation.bin`.
Instead of storing the original text, we save the processed token IDs in binary files. This allows us to load the training data much faster during model training.

### (3) Store the processed data on disk.
Keeping the token IDs in binary files on disk avoids repeatedly tokenizing the dataset and reduces RAM usage, making training more efficient, especially for large datasets.

In [ ]:
!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

# here we are using gpt2 tokenizer as gpt4 or gpt5 tokenizer would take more space there would'nt be much of a difference.
enc = tiktoken.get_encoding("gpt2")

def process(example):
    # Convert the text into token IDs without adding any special tokens.
    ids = enc.encode_ordinary(example['text'])
    # Add <|endoftext|> token (50256) at the end of each story so the model learns story boundaries.
    ids.append(enc.eot_token) # 50256
    out = {'ids': ids, 'len': len(ids)}
    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map( #maps the stories one by one to the process function
        process,
        remove_columns=['text'], # since the stories are converted into token ids so text inside them is not needed as it will take extra space on ram
        desc="tokenizing the splits",
        num_proc=8, # uses all 8 cores of cpu to tokenization simultaneosly 
        )
    
    # Combine all token IDs into a single binary file for efficient training.
    for split, dset in tokenized.items(): # same code works for both datasets without writing it twice.
        arr_len = np.sum(dset['len'], dtype=np.uint64) # used to calculate total token ids nd stores the result as 64bit integer 
        filename = f'{split}.bin'

        dtype = np.uint16

        # Create a memory-mapped binary file to store the token IDs on disk.
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):

            # Process the dataset in batches for faster disk writes.
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')

            arr_batch = np.concatenate(batch['ids'])

            # Write the current batch of token IDs into the binary file.
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        # Ensure all data is written from memory to disk.
        arr.flush()

In [14]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


# Generate input-output pairs from the dataset

In [15]:
import torch
import numpy as np


# block_size = context window (the number of previous tokens the model sees)

# Use the Apple Silicon GPU (MPS) only.
if not torch.backends.mps.is_available():
    raise RuntimeError("Apple Silicon GPU (MPS) is not available.")

device = "mps"
device_type = "mps"


def get_batch(split):
    # Reload the memory-mapped file for every batch.
    # This prevents a known memory leak when using np.memmap.
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('validation.bin', dtype=np.uint16, mode='r')

    # Randomly choose the starting position for each sequence.
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Input sequence (context)
    x = torch.stack([
        torch.from_numpy((data[i:i + block_size]).astype(np.int64))
        for i in ix
    ])

    # Target sequence (same sequence shifted by one token)
    y = torch.stack([
        torch.from_numpy((data[i + 1:i + 1 + block_size]).astype(np.int64))
        for i in ix
    ])

    # Move the batch to the Apple Silicon GPU.
    x = x.to(device)
    y = y.to(device)

    return x, y

# Configure the SLM architecture

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [17]:
config = GPTConfig(
    vocab_size=50257,     # use the tokenizer's vocab size
    block_size=128,       # or whatever context size you're training with
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1,
    bias=True
)

device = torch.device("mps")
model = GPT(config).to(device)

In [18]:
print(next(model.parameters()).device)

mps:0


In [19]:
print(torch.backends.mps.is_available())

True


#  Define the loss function

In [20]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

# Define SLM Training Configuration 

In [21]:
# Training Config
import torch
from contextlib import nullcontext

learning_rate = 3e-4 # standard for small GPT models
max_iters = 5000 # reduced: with grad_accum=4, we get 1250 optimizer steps in ~4x less wall time
warmup_steps = 200 # ~4% of max_iters
min_lr = 1e-5 # must be LOWER than learning_rate for cosine decay to work
eval_iters = 250 # evaluate more frequently
batch_size = 32 # micro-batch size
block_size = 128 # context window

gradient_accumulation_steps = 4 # effective batch = 32*4 = 128 (sensible for ~29M params)

device = "mps"
device_type = "mps"
# note: float16 data type will automatically use a GradScaler
assert torch.backends.mps.is_available(), "MPS GPU not available on this machine"
dtype = "bfloat16"
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

# Enable mixed precision — this was previously nullcontext(), meaning bfloat16 was never actually used
ctx = torch.amp.autocast(device_type="mps", dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

# Define SLM Training Configuration Part 2

In [22]:
import torch
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

##PUT IN WEIGHT DECAY, CHANGED BETA2 to 0.95
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9) #weight decay for regularization

scheduler_warmup = LinearLR(optimizer, total_iters=warmup_steps) #Implement linear warmup
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=min_lr) #Implement lr decay
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps]) #Switching from warmup to decay


scaler = torch.amp.GradScaler("mps", enabled=(dtype == 'float16'))

In [23]:
X, y = get_batch("train")
X, y = X.to(device), y.to(device)

print(X.device)
print(y.device)

mps:0
mps:0


In [24]:
import torch

print(torch.__version__)
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

2.13.0
True
True


# Pre-train the SLM

In [ ]:
best_val_loss = float("inf")
best_model_params_path = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []


model = model.to(device)


for epoch in tqdm(range(max_iters)):
    if epoch % eval_iters == 0 and epoch != 0:
        
        losses = estimate_loss(model)
        print(f"Epoch {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"The current learning rate: {optimizer.param_groups[0]['lr']:.5f}")
        train_loss_list += [losses['train']]
        validation_loss_list += [losses['val']]

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    
    X, y = get_batch("train")
    X, y = X.to(device), y.to(device)

    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step() # step scheduler per optimizer step, not per micro-batch

In [25]:
from pathlib import Path

print(Path("best_model_params.pt").exists())

True


In [26]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [27]:
model.load_state_dict(torch.load("best_model_params.pt", map_location=device))
model.eval()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 384)
    (wpe): Embedding(128, 384)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=384, out_features=1152, bias=True)
          (c_proj): Linear(in_features=384, out_features=384, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=384, out_features=1536, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1536, out_features=384, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=384, out_features=50257, bias=False)
)

# Plot the SLM Loss Function

In [28]:
import matplotlib.pyplot as plt
train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]
validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]

plt.plot(train_loss_list_converted, 'g', label='train_loss')
plt.plot(validation_loss_list_converted, 'r', label='validation_loss')
plt.xlabel("Steps - Every 100 epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()



NameError: name 'train_loss_list' is not defined

# Perform inference using the trained SLM

In [33]:
# Load the best model
model = GPT(config)  # re-create the model with same config
device = "mps" if torch.backends.mps.is_available() else "cpu"
best_model_params_path = "best_model_params.pt"
model.load_state_dict(torch.load(best_model_params_path, map_location=torch.device(device)))
model = model.to(device)
model.eval()

# Generate text
import tiktoken
enc = tiktoken.get_encoding("gpt2")

prompt = "There was a little girl named"
input_ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)

with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=200, temperature=0.8, top_k=40)

print(enc.decode(output[0].tolist()))


There was a little girl named Lily. She loved to play with her toys and pretend to be a little girl named Lily. One day, Lily's mom came to visit her. She saw Lily and wanted to help her.

Lily's mom saw the doll and said, "You can't play with it, but it can't you play with it." Lily was sad because she wanted the doll. But her mom was sad because Lily missed her toys.

"I have no friends," said Lily. "Are you lost my doll?"

Her mom saw her and asked, "Lily, it's okay?" Lily said, "I'm okay, Lily. I'll be your doll."

But Lily was also happy. She liked the doll too. She did not like the doll anymore. She wanted to play with her doll's doll, but she was scared. She saw Lily and her doll. She was sad and sad. She said, "What is wrong, Lily?"



In [38]:
model.load_state_dict(torch.load("best_model_params.pt", map_location=device))
model = model.to(device)

In [40]:
losses = estimate_loss(model)
print("Train loss:", losses['train'].item())
print("Val loss:", losses['val'].item())

Train loss: 2.422661542892456
Val loss: 2.4321177005767822


In [45]:
import math
print("Perplexity:", math.exp(2.4))

Perplexity: 11.023176380641601


In [42]:
model = GPT(config)
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.load_state_dict(torch.load("best_model_params.pt", map_location=torch.device(device)))
model = model.to(device)
model.eval()

import tiktoken
enc = tiktoken.get_encoding("gpt2")

prompts = [
    "There was a little girl named",
    "Once upon a time, a dog",
    "The sun was shining and",
    "Tom and Lily went to the",
]

for p in prompts:
    input_ids = torch.tensor([enc.encode(p)], dtype=torch.long, device=device)
    out = model.generate(input_ids, max_new_tokens=100)
    print(enc.decode(out[0].tolist()))
    print("-" * 50)

There was a little girl named Lily. She had a big hat that she loved to ride it around in. One day, Lily went outside to playtime and she saw a serious task. "Let's go!" said Lily. "No, I don't want to get safe."

Lily's mom said, "But we have to be careful," said Lily. "Okay, we do something else." She stubborn to do what Lily said. She said, "It's not nice, Lily. The tornado is
--------------------------------------------------
Once upon a time, a dog named Spot went for a walk in the park. Spot was a very tight and she loved to go outside.

They would pretend he was more excited and of going out playing in the grass. One day, Spot came into the park and he saw a boy. The boy wanted to sing so much. He thought hard would be like to come with him.

"No, Spot," said Ralph. "You can pray, honey. But I can fly."

Spot did not
--------------------------------------------------
The sun was shining and the sun was shining brightly at night. Lily smiled and knew what her mom had was picked u

In [43]:
sum(p.numel() for p in model.parameters())

29995392